# Hexagonal Image Representation for Retinal Vessel Segmentation

## Objective
This project explores the application of hexagonal image representation on retinal images and evaluates its impact on segmentation performance.

We compare:
- Square vs Hex image representations
- Segmentation methods: Region Growing and Otsu Thresholding
- Evaluation using Ground Truth masks (Dice Score)

In [ ]:
import os

for root, dirs, files in os.walk("DRIVE"):
    print(root)

## Dataset

We use the DRIVE dataset, which contains:
- Retinal images
- Ground truth vessel segmentation masks

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!unzip archive.zip

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


## Sample Image Visualization

In [ ]:
img = cv2.imread("DRIVE/training/images/21_training.tif", cv2.IMREAD_GRAYSCALE)
mask = cv2.imread("DRIVE/training/1st_manual/21_manual1.gif", cv2.IMREAD_GRAYSCALE)

if img is None or mask is None:
    raise ValueError("Image or Mask not found")

# resize (important for consistency)
img = cv2.resize(img, (256, 256))
mask = cv2.resize(mask, (256, 256))

# normalize
img = img.astype(np.float32) / 255.0
mask = mask.astype(np.float32) / 255.0

# display
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(img, cmap='gray')
plt.title("Retinal Image (Input)")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(mask, cmap='gray')
plt.title("Ground Truth (Vessels)")
plt.axis("off")

plt.show()

## Hexagonal Image Representation

We approximate hexagonal sampling using row-wise pixel shifting.

In [ ]:
def square_to_hex(img):
    h, w = img.shape
    hex_img = np.zeros_like(img)

    for i in range(h):
        for j in range(w):
            # shift alternate rows to simulate hex grid
            if i % 2 == 1:
                new_j = j + 1
            else:
                new_j = j

            if new_j < w:
                hex_img[i, j] = img[i, new_j]
            else:
                hex_img[i, j] = img[i, j]

    return hex_img

In [ ]:
hex_img = square_to_hex(img)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(hex_img, cmap='gray')
plt.title("Hexagonally Resampled Image")
plt.axis("off")

plt.show()

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

psnr_value = psnr(img, hex_img)
ssim_value = ssim(img, hex_img, data_range=1.0)

print(f"PSNR: {psnr_value:.4f}")
print(f"SSIM: {ssim_value:.4f}")

In [ ]:
HEX_NEIGHBORS = [
    (-1, 0), (-1, 1),
    (0, -1), (0, 1),
    (1, -1), (1, 0)
]

SQUARE_4_NEIGHBORS = [
    (-1, 0), (1, 0),
    (0, -1), (0, 1)
]


## Region Growing Segmentation

We perform segmentation using:
- Square connectivity (4-neighbors)
- Hex connectivity (6-neighbors)

In [ ]:
def region_growing(img, seed, neighbors, threshold=0.08):
    h, w = img.shape
    seg = np.zeros_like(img)
    visited = np.zeros_like(img, dtype=bool)

    seed_val = img[seed]
    stack = [seed]

    while stack:
        x, y = stack.pop()

        # safety check (important)
        if not (0 <= x < h and 0 <= y < w):
            continue

        if visited[x, y]:
            continue

        visited[x, y] = True

        # threshold condition
        if abs(img[x, y] - seed_val) < threshold:
            seg[x, y] = 1

            for dx, dy in neighbors:
                nx, ny = x + dx, y + dy

                # push only valid & unvisited pixels
                if 0 <= nx < h and 0 <= ny < w and not visited[nx, ny]:
                    stack.append((nx, ny))

    return seg

In [ ]:
# get multiple bright points (likely vessels)
seeds = np.argwhere(img > 0.7)

# initialize as integer (IMPORTANT)
seg_sq = np.zeros_like(img, dtype=np.uint8)
seg_hex = np.zeros_like(img, dtype=np.uint8)

for s in seeds[:50]:   # limit for speed
    s = tuple(s)

    out_sq  = region_growing(img, s, SQUARE_4_NEIGHBORS, threshold=0.1)
    out_hex = region_growing(hex_img, s, HEX_NEIGHBORS, threshold=0.1)

    # convert to binary before OR
    seg_sq  |= (out_sq > 0).astype(np.uint8)
    seg_hex |= (out_hex > 0).astype(np.uint8)

In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(seg_sq, cmap='gray')
plt.title("Square Segmentation")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(seg_hex, cmap='gray')
plt.title("Hex Segmentation")
plt.axis("off")

plt.show()

In [ ]:
def extract_boundary(seg, neighbors):
    h, w = seg.shape
    boundary = np.zeros_like(seg)

    for i in range(h):
        for j in range(w):
            if seg[i, j] == 1:

                is_boundary = False

                for dx, dy in neighbors:
                    ni, nj = i + dx, j + dy

                    # if neighbor is out of bounds OR background → boundary
                    if not (0 <= ni < h and 0 <= nj < w) or seg[ni, nj] == 0:
                        is_boundary = True
                        break

                if is_boundary:
                    boundary[i, j] = 1

    return boundary

In [ ]:
hex_boundary = extract_boundary(seg_hex, HEX_NEIGHBORS)
sq_boundary  = extract_boundary(seg_sq,  SQUARE_4_NEIGHBORS)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(sq_boundary, cmap='gray')
plt.title("Square Boundary")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(hex_boundary, cmap='gray')
plt.title("Hex Boundary")
plt.axis("off")

plt.show()

In [ ]:
zoom_hex = seg_hex[60:120, 80:140]
zoom_sq  = seg_sq[60:120, 80:140]
zoom_img = img[60:120, 80:140]

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(zoom_img, cmap='gray')
plt.title("Original (Zoomed)")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(zoom_sq, cmap='gray')
plt.title("Square (Zoomed)")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(zoom_hex, cmap='gray')
plt.title("Hex (Zoomed)")
plt.axis("off")

plt.show()

## Region Growing Evaluation (Dice Score)

In [ ]:
def dice_score(pred, target):
    pred = (pred > 0.5).astype(np.float32)   # convert to binary

    intersection = np.sum(pred * target)
    return (2 * intersection) / (np.sum(pred) + np.sum(target) + 1e-8)

In [ ]:
dice_sq = dice_score(seg_sq, mask)
dice_hex = dice_score(seg_hex, mask)

print("Dice Score (Square):", dice_sq)
print("Dice Score (Hex):", dice_hex)

In [ ]:
print(f"Dice Score (Square): {dice_sq:.4f}")
print(f"Dice Score (Hex): {dice_hex:.4f}")

In [ ]:
sample = "21_training.tif"

orig = cv2.imread(f"DRIVE/training/images/{sample}", 0)
hexv = cv2.imread(f"HEX_DRIVE/training/images/{sample}", 0)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(orig, cmap='gray')
plt.title("Original")

plt.subplot(1,2,2)
plt.imshow(hexv, cmap='gray')
plt.title("Hex Dataset Image")

plt.show()

In [ ]:
!pip install scikit-image

In [ ]:
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

## Otsu Thresholding with Preprocessing

To improve segmentation:
- Contrast enhancement (CLAHE)
- Vessel enhancement (Black-hat)
- ROI masking

In [ ]:
def normalize(img):
    img = img.astype(np.float32)
    img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-8)
    return img

In [ ]:
def preprocess(img):
    img_uint8 = (img * 255).astype(np.uint8)

    # CLAHE (contrast enhancement)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(img_uint8)

    # Black-hat (highlights dark vessels)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15,15))
    blackhat = cv2.morphologyEx(enhanced, cv2.MORPH_BLACKHAT, kernel)

    return blackhat


In [ ]:
def otsu_segmentation(img, mask):
    # normalize first
    img = normalize(img)

    # preprocessing
    proc = preprocess(img)

    # smoothing (stabilizes Otsu)
    proc = cv2.GaussianBlur(proc, (3,3), 0)

    # apply mask
    proc = proc * (mask > 0)

    # Otsu threshold
    _, thresh = cv2.threshold(
        proc,
        0,
        255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    seg = thresh / 255.0
    seg = seg * (mask > 0)

    return seg

In [ ]:
seg_sq_otsu = otsu_segmentation(img, mask)
seg_hex_otsu = otsu_segmentation(hex_img, mask)

In [ ]:
plt.figure(figsize=(15,4))

plt.subplot(1,4,1)
plt.imshow(img, cmap='gray')
plt.title("Original")
plt.axis("off")

plt.subplot(1,4,2)
plt.imshow(seg_sq_otsu, cmap='gray')
plt.title("Square Otsu")
plt.axis("off")

plt.subplot(1,4,3)
plt.imshow(seg_hex_otsu, cmap='gray')
plt.title("Hex Otsu")
plt.axis("off")

plt.subplot(1,4,4)
plt.imshow(mask, cmap='gray')
plt.title("Ground Truth")
plt.axis("off")

plt.show()

In [ ]:
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(seg_sq_otsu * mask, cmap='gray')
plt.title("Square ∩ Ground Truth")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(seg_hex_otsu * mask, cmap='gray')
plt.title("Hex ∩ Ground Truth")
plt.axis("off")

plt.show()

## Otsu Evaluation (Dice Score)

In [ ]:
# force everything to binary (IMPORTANT)
seg_sq_otsu = (seg_sq_otsu > 0.5).astype(np.float32)
seg_hex_otsu = (seg_hex_otsu > 0.5).astype(np.float32)
mask = (mask > 0).astype(np.float32)

In [ ]:
dice_sq = dice_score(seg_sq_otsu, mask)
dice_hex = dice_score(seg_hex_otsu, mask)

print(f"Otsu Dice (Square): {dice_sq:.4f}")
print(f"Otsu Dice (Hex): {dice_hex:.4f}")

Creation of Hex Dataset , give square image (DRIVE) dataset is converted to an hexogonal imagge dataset.

In [ ]:
!rm -rf HEX_DRIVE

In [ ]:
import os

os.makedirs("HEX_DRIVE/training/images", exist_ok=True)
os.makedirs("HEX_DRIVE/training/masks", exist_ok=True)

In [ ]:
import cv2
import numpy as np
import os

img_dir = "DRIVE/training/images"
mask_dir = "DRIVE/training/1st_manual"

# create folders (safe)
os.makedirs("HEX_DRIVE/training/images", exist_ok=True)
os.makedirs("HEX_DRIVE/training/masks", exist_ok=True)

img_files = sorted(os.listdir(img_dir))

count = 0

for img_name in img_files:

    try:
        # ---- READ IMAGE ----
        img_path = os.path.join(img_dir, img_name)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if img is None:
            print(f"Skipping (image error): {img_name}")
            continue

        img = cv2.resize(img, (256, 256))
        img = img.astype(np.float32) / 255.0

        # ---- HEX CONVERSION ----
        hex_img = square_to_hex(img)

        # ---- SAVE HEX IMAGE (as PNG) ----
        hex_save = (hex_img * 255).astype(np.uint8)

        img_name_png = img_name.replace(".tif", ".png")
        save_img_path = os.path.join("HEX_DRIVE/training/images", img_name_png)
        cv2.imwrite(save_img_path, hex_save)

        # ---- HANDLE MASK ----
        mask_name = img_name.replace("_training.tif", "_manual1.gif")
        mask_path = os.path.join(mask_dir, mask_name)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if mask is None:
            print(f"Skipping (mask error): {mask_name}")
            continue

        mask = cv2.resize(mask, (256, 256))

        # ---- SAVE MASK AS PNG (FIX) ----
        mask_name_png = mask_name.replace(".gif", ".png")
        mask_save_path = os.path.join("HEX_DRIVE/training/masks", mask_name_png)
        cv2.imwrite(mask_save_path, mask)

        count += 1

    except Exception as e:
        print(f"Error in {img_name}: {e}")

print(f"✅ Total images processed: {count}")

In [ ]:
print("Hex images:", len(os.listdir("HEX_DRIVE/training/images")))
print("Masks:", len(os.listdir("HEX_DRIVE/training/masks")))

In [ ]:
sample = os.listdir("HEX_DRIVE/training/masks")[0]

mask = cv2.imread(f"HEX_DRIVE/training/masks/{sample}", 0)

plt.imshow(mask, cmap='gray')
plt.title("Mask Check")
plt.show()

In [ ]:
!zip -r HEX_DRIVE.zip HEX_DRIVE

In [ ]:
from google.colab import files
files.download("HEX_DRIVE.zip")

## Region Growing Evaluation (Dice Score)(Ground Truth)

In [ ]:
seg_sq = (seg_sq > 0).astype(np.float32)
seg_hex = (seg_hex > 0).astype(np.float32)

In [ ]:
mask_bin = (mask > 0).astype(np.float32)

In [ ]:
dice_sq_rg = dice_score(seg_sq, mask_bin)
dice_hex_rg = dice_score(seg_hex, mask_bin)

print(f"Region Growing Dice (Square): {dice_sq_rg:.4f}")
print(f"Region Growing Dice (Hex): {dice_hex_rg:.4f}")

In [ ]:
plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(seg_sq * mask_bin, cmap='gray')
plt.title("Square RG ∩ Ground Truth")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(seg_hex * mask_bin, cmap='gray')
plt.title("Hex RG ∩ Ground Truth")
plt.axis("off")

plt.show()

In [ ]:
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(seg_sq, cmap='gray')
plt.title("Square RG")

plt.subplot(1,3,2)
plt.imshow(seg_hex, cmap='gray')
plt.title("Hex RG")

plt.subplot(1,3,3)
plt.imshow(mask_bin, cmap='gray')
plt.title("Ground Truth")

plt.show()

## Observations

- Hex representation improves connectivity-based segmentation (Region Growing)
- For intensity-based methods like Otsu, results vary
- Preprocessing significantly improves Otsu performance

## Results Summary

| Method            | Square Dice | Hex Dice | Improvement |
|------------------|------------|----------|-------------|
| Region Growing   | 0.1153     | 0.2367   | +0.1214     |
| Otsu Threshold   | 0.6567     | 0.6833   | +0.0266     |

## Conclusion

Hexagonal image representation improves segmentation performance in methods that rely on neighborhood connectivity. While its impact on global thresholding methods is inconsistent, it enhances structural continuity in vessel detection.

Overall, hex representation provides a promising alternative to traditional square pixel grids in image processing tasks.

## Future Work

- Apply hex representation to deep learning models
- Explore advanced interpolation methods
- Evaluate on larger datasets